# 02 — Passenger Demand Estimation
## AI City Twin Casablanca

Score routes by population coverage and nearby POIs.

In [ ]:
import sys
sys.path.insert(0, '..')

from shapely.geometry import LineString
import matplotlib.pyplot as plt
from src.data_prep import generate_synthetic_pois, generate_population_grid
from src.demand_model import DemandModel

In [ ]:
# Initialize model
pois = generate_synthetic_pois()
pop_grid = generate_population_grid()
model = DemandModel(pop_grid=pop_grid, pois=pois)

print(f'Population total: {pop_grid["population"].sum():,}')
print(f'POIs total: {len(pois)}')

In [ ]:
# Define sample routes
routes = [
    {'route_id': 'East-West Central',
     'geometry': LineString([(-7.63, 33.58), (-7.59, 33.57), (-7.55, 33.56), (-7.52, 33.55)])},
    {'route_id': 'North-South',
     'geometry': LineString([(-7.59, 33.62), (-7.58, 33.59), (-7.57, 33.56), (-7.56, 33.53)])},
    {'route_id': 'Peripheral Ring',
     'geometry': LineString([(-7.67, 33.57), (-7.65, 33.60), (-7.60, 33.62), (-7.55, 33.60)])},
]

# Score all routes
scores_df = model.score_multiple_routes(routes)
scores_df[['route_id', 'total_score', 'covered_population', 'total_pois', 'route_length_km', 'score_per_km', 'rank']]

In [ ]:
# Visualize coverage for top route
top_route = routes[0]['geometry']
coverage = model.generate_coverage_map(top_route)

fig, ax = plt.subplots(1, 1, figsize=(12, 10))
pop_grid.plot(ax=ax, column='population', cmap='YlOrRd', alpha=0.4, legend=True)
coverage['catchment'].plot(ax=ax, color='blue', alpha=0.15)
if 'covered_pois' in coverage:
    coverage['covered_pois'].plot(ax=ax, color='green', markersize=15)

import geopandas as gpd
gpd.GeoDataFrame(geometry=[top_route], crs='EPSG:4326').plot(ax=ax, color='red', linewidth=3)
ax.set_title('Route Coverage Analysis')
plt.tight_layout()
plt.show()